# Exercise: An AI-Powered SQL Agent

Build an agent that answers questions about a real dataset by writing and running SQL, using the raw Groq API and the hand-written agentic loop from notebook 1. The data is the January 2026 NYC yellow taxi trips, queried through DuckDB.

Fill in each `# YOUR CODE HERE`. A fully worked solution is in [`solutions/2_exercise_sql_agent.ipynb`](solutions/2_exercise_sql_agent.ipynb).

## Learning Objectives

At the end of this exercise, you should be able to:

- Build a SQL agent from scratch that inspects a schema and runs queries through tools.
- Test an agent by asserting on its answer and on the order of its tool calls.
- Evaluate open-ended answers with a second model acting as a judge.
- Track the token usage and estimated cost of a run.

## Setup

We use the raw Groq client and DuckDB. `load_dotenv()` reads `GROQ_API_KEY` from your local `.env` file. This cell is given.

In [1]:
import json
import os
import urllib.request
from collections.abc import Callable

import duckdb
from dotenv import load_dotenv
from groq import Groq
from groq.types.chat import ChatCompletionMessageParam, ChatCompletionToolParam

In [2]:
load_dotenv()

client = Groq()
MODEL = "openai/gpt-oss-20b"

## Load the data (given)

We download one month of NYC yellow taxi trips as a Parquet file (about 64 MB) and load it into a DuckDB table named `trips`. This cell is given.

In [3]:
DATA_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"
)
PARQUET_FILE = "yellow_tripdata_2026-01.parquet"

if not os.path.exists(PARQUET_FILE):
    print("Downloading the taxi data (about 64 MB) ...")
    urllib.request.urlretrieve(DATA_URL, PARQUET_FILE)

con = duckdb.connect("taxi.db")
con.execute(f"CREATE TABLE IF NOT EXISTS trips AS SELECT * FROM '{PARQUET_FILE}'")
row_count_result = con.execute("SELECT COUNT(*) FROM trips").fetchone()
if row_count_result is None:
    raise RuntimeError("The row-count query returned no result.")
row_count = row_count_result[0]
row_count

3724889

## Task 1: the tools

Implement the two tools the agent will use. `get_schema` returns the columns and types of the `trips` table, and `run_sql` runs a query and returns up to 50 result rows. The agent cannot touch the database itself: it can only ask us to call these functions.

In [4]:
def get_schema() -> str:
    """Return the columns and types of the trips table."""
    # YOUR CODE HERE
    # Hint: run "DESCRIBE trips" with con.execute(...).fetchall(), then format
    # each row as "name: type".
    raise NotImplementedError("Complete Task 1: implement get_schema().")


def run_sql(query: str) -> str:
    """Run a SQL query against the trips table and return up to 50 result rows."""
    # YOUR CODE HERE
    # Hint: str(con.sql(query).limit(50)) renders a readable table.
    raise NotImplementedError("Complete Task 1: implement run_sql().")


def parse_tool_arguments(raw_arguments: str | None) -> dict[str, object]:
    """Parse a tool-call JSON object and reject other JSON values."""
    arguments = json.loads(raw_arguments or "{}")
    if not isinstance(arguments, dict):
        raise TypeError("Tool arguments must be a JSON object.")
    return arguments


tool_functions: dict[str, Callable[..., object]] = {
    "get_schema": get_schema,
    "run_sql": run_sql,
}

In [10]:
def get_schema() -> str:
    """Return the columns and types of the trips table."""

    rows = con.execute("DESCRIBE trips").fetchall()

    return "\n".join(
        f"{row[0]}: {row[1]}"
        for row in rows
    )

In [11]:
print(get_schema())

VendorID: INTEGER
tpep_pickup_datetime: TIMESTAMP
tpep_dropoff_datetime: TIMESTAMP
passenger_count: BIGINT
trip_distance: DOUBLE
RatecodeID: BIGINT
store_and_fwd_flag: VARCHAR
PULocationID: INTEGER
DOLocationID: INTEGER
payment_type: BIGINT
fare_amount: DOUBLE
extra: DOUBLE
mta_tax: DOUBLE
tip_amount: DOUBLE
tolls_amount: DOUBLE
improvement_surcharge: DOUBLE
total_amount: DOUBLE
congestion_surcharge: DOUBLE
Airport_fee: DOUBLE
cbd_congestion_fee: DOUBLE


## Task 2: describe the tools to the model

Describe both tools to the model as JSON schemas, the same shape you saw in notebook 1. `get_schema` takes no arguments; `run_sql` takes a single `query` string.

In [ ]:
tools: list[ChatCompletionToolParam] = [
    # YOUR CODE HERE
]

In [12]:
tools: list[ChatCompletionToolParam] = [
    {
        "type": "function",
        "function": {
            "name": "get_schema",
            "description": "Return the schema of the DuckDB table named trips.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_sql",
            "description": "Run a SQL query against the DuckDB table named trips and return the query result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": "The SQL query to execute against the trips table.",
                    }
                },
                "required": ["sql"],
                "additionalProperties": False,
            },
        },
    },
]

## Task 3: the agent loop

Build the agentic loop, reusing the pattern from notebook 1. The system prompt is given. In the loop, add up the token usage the API reports (`response.usage.prompt_tokens` and `response.usage.completion_tokens`) and record the name of every tool the model calls, so the tests below can inspect them.

In [13]:
SYSTEM_PROMPT = (
    "You are a SQL analyst for a DuckDB table named trips. "
    "Always call get_schema first to inspect the columns, then write and run one "
    "SQL query with run_sql, and finally answer the question using the query result. "
    "Be concise."
)


def run_sql_agent(question: str) -> dict:
    messages: list[ChatCompletionMessageParam] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    tool_calls_made = []
    prompt_tokens = completion_tokens = 0

    # YOUR CODE HERE
    # Loop, as in notebook 1:
    #  - call client.chat.completions.create(model=MODEL, temperature=0,
    #    messages=messages, tools=tools)
    #  - add response.usage.prompt_tokens / completion_tokens to the counters
    #  - if the reply has no tool_calls, append it and stop
    #  - otherwise append the reply, parse its arguments with
    #    parse_tool_arguments(), run each requested tool, and append a
    #    {"role": "tool", "tool_call_id": ..., "content": ...} message,
    #    and record each tool name in tool_calls_made

    return {
        "answer": messages[-1]["content"],
        "tool_calls": tool_calls_made,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
    }

In [14]:
SYSTEM_PROMPT = (
    "You are a SQL analyst for a DuckDB table named trips. "
    "Always call get_schema first to inspect the columns, then write and run one "
    "SQL query with run_sql, and finally answer the question using the query result. "
    "Be concise."
)

def run_sql_agent(question: str) -> dict:
    messages: list[ChatCompletionMessageParam] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    tool_calls_made = []
    prompt_tokens = 0
    completion_tokens = 0

    while True:

        # --------------------------------------------------
        # Ask the model what to do next
        # --------------------------------------------------
        response = client.chat.completions.create(
            model=MODEL,
            temperature=0,
            messages=messages,
            tools=tools,
        )

        # --------------------------------------------------
        # Track token usage
        # --------------------------------------------------
        if response.usage:
            prompt_tokens += response.usage.prompt_tokens
            completion_tokens += response.usage.completion_tokens

        message = response.choices[0].message

        # --------------------------------------------------
        # No tool call -> final answer
        # --------------------------------------------------
        if not message.tool_calls:

            messages.append({
                "role": "assistant",
                "content": message.content or "",
            })

            break

        # --------------------------------------------------
        # Add assistant's tool-call message
        # --------------------------------------------------
        messages.append(message)

        # --------------------------------------------------
        # Execute each requested tool
        # --------------------------------------------------
        for tool_call in message.tool_calls:

            tool_name = tool_call.function.name
            tool_arguments = parse_tool_arguments(
                tool_call.function.arguments
            )

            tool_calls_made.append(tool_name)

            # ----------------------------------------------
            # Run get_schema
            # ----------------------------------------------
            if tool_name == "get_schema":

                result = get_schema(**tool_arguments)

            # ----------------------------------------------
            # Run SQL query
            # ----------------------------------------------
            elif tool_name == "run_sql":

                result = run_sql(**tool_arguments)

            # ----------------------------------------------
            # Unknown tool
            # ----------------------------------------------
            else:

                result = f"Unknown tool: {tool_name}"

            # ----------------------------------------------
            # Send tool result back to the model
            # ----------------------------------------------
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result),
            })

    # ------------------------------------------------------
    # Return final result
    # ------------------------------------------------------
    return {
        "answer": messages[-1]["content"],
        "tool_calls": tool_calls_made,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
    }

## Ask the agent a question

Once the loop works, this should answer the question by inspecting the schema and writing its own SQL.

In [16]:
def run_sql(sql: str) -> str:
    """Execute a SQL query against the DuckDB trips table."""

    result = con.execute(sql)

    rows = result.fetchall()

    if not rows:
        return "Query returned no rows."

    columns = [desc[0] for desc in result.description]

    output = [" | ".join(columns)]

    for row in rows:
        output.append(
            " | ".join(str(value) for value in row)
        )

    return "\n".join(output)

In [17]:
result = run_sql_agent("How many trips had more than 5 passengers?")
result["answer"]

'There were **4,894** trips that had more than 5 passengers.'

## Task 4: assert the answer is correct

Compute the true value directly from the database, then assert that the agent's answer contains it. This catches wrong SQL and wrong arithmetic.

In [ ]:
expected_result = con.execute(
    "SELECT COUNT(*) FROM trips WHERE passenger_count > 5"
).fetchone()
if expected_result is None:
    raise RuntimeError("The validation query returned no result.")
expected = expected_result[0]


if expected_result is None:
    raise RuntimeError("The validation query returned no result.")

expected = expected_result[0]

# Normalize the agent answer by removing thousands separators
answer_text = result["answer"].replace(",", "")

# Check that the expected count appears in the agent's answer
if str(expected) not in answer_text:
    raise AssertionError(
        f"Expected count {expected} not found in agent answer:\n"
        f"{result['answer']}"
    )

print(f"✅ PASS: The expected count ({expected}) appears in the agent answer.")

✅ PASS: The expected count (4894) appears in the agent answer.


## Task 5: assert the tool-call order

A correct answer is not enough. The agent should inspect the schema before it writes SQL, so it does not guess column names. Assert that the first tool call was `get_schema` and that `run_sql` was used.

In [21]:
# YOUR CODE HERE
# assert result["tool_calls"][0] == "get_schema" and "run_sql" in result["tool_calls"]


assert (
    result["tool_calls"][0] == "get_schema"
    and "run_sql" in result["tool_calls"]
), f"Unexpected tool call sequence: {result['tool_calls']}"

print("✅ PASS: get_schema was called first and run_sql was called.")

✅ PASS: get_schema was called first and run_sql was called.


## Task 6: an LLM as a judge

Some answers cannot be checked with an exact match, for example a one-sentence explanation. Write a `judge` function that asks a second model to grade an answer against a natural-language criterion and reply with only `PASS` or `FAIL`.

In [ ]:
def judge(question: str, answer: str, criteria: str) -> str:
    """Ask a second model whether the answer meets a natural-language criterion."""
    # YOUR CODE HERE
    # Send a chat completion whose system prompt tells the grader to reply only
    # PASS or FAIL, and whose user message contains the question, answer, and
    # criteria. Return the reply text.
    raise NotImplementedError("Complete Task 5: implement judge().")

In [22]:
def judge(question: str, answer: str, criteria: str) -> str:
    """Ask a second model whether the answer meets a natural-language criterion."""

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a strict evaluator. "
                    "Evaluate whether the answer satisfies the given criterion. "
                    "Reply with ONLY one word: PASS or FAIL."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Question:\n{question}\n\n"
                    f"Answer:\n{answer}\n\n"
                    f"Criterion:\n{criteria}"
                ),
            },
        ],
    )

    return response.choices[0].message.content.strip()

## Task 7: several scenarios

Real test suites cover more than one case. Add a few scenarios, each pairing a question with the criterion its answer must meet, then run the agent on each and let the judge grade it.

In [ ]:
scenarios = [
    # YOUR CODE HERE: add {"question": ..., "criteria": ...} dicts,
    # for example the average trip distance or the total number of trips.
]

for case in scenarios:
    run = run_sql_agent(case["question"])
    verdict = judge(case["question"], run["answer"], case["criteria"])
    print(f"[{verdict}] {case['question']}")
    print("   ->", run["answer"])
    print("   tools:", run["tool_calls"])

In [23]:
scenarios = [

    {
        "question": "What is the average trip distance?",
        "criteria": (
            "The answer must provide the average trip distance calculated "
            "from the trips table."
        ),
    },

    {
        "question": "How many trips are in the trips table?",
        "criteria": (
            "The answer must provide the total number of trips in the "
            "trips table."
        ),
    },

    {
        "question": "What is the maximum number of passengers in a trip?",
        "criteria": (
            "The answer must provide the maximum passenger_count value "
            "from the trips table."
        ),
    },

    {
        "question": "How many trips had more than 5 passengers?",
        "criteria": (
            "The answer must provide the correct number of trips where "
            "passenger_count is greater than 5."
        ),
    },
]


for case in scenarios:

    run = run_sql_agent(case["question"])

    verdict = judge(
        case["question"],
        run["answer"],
        case["criteria"]
    )

    print(f"[{verdict}] {case['question']}")
    print("   ->", run["answer"])
    print("   tools:", run["tool_calls"])

[PASS] What is the average trip distance?
   -> The average trip distance is **approximately 6.46 units**.
   tools: ['get_schema', 'run_sql']
[PASS] How many trips are in the trips table?
   -> There are **3,724,889** trips in the `trips` table.
   tools: ['get_schema', 'run_sql']
[PASS] What is the maximum number of passengers in a trip?
   -> The maximum number of passengers on a single trip is **9**.
   tools: ['get_schema', 'run_sql']
[FAIL] How many trips had more than 5 passengers?
   -> There were **4,894** trips with more than 5 passengers.
   tools: ['get_schema', 'run_sql']


## Task 8: track the cost

Every response reports how many tokens it used, which you summed in the loop. Groq bills per token, so turn the counts into an estimated cost. The rates below are approximate and change over time.

In [ ]:
INPUT_RATE = 0.075 / 1_000_000  # USD per input token (approximate)
OUTPUT_RATE = 0.30 / 1_000_000  # USD per output token (approximate)

# YOUR CODE HERE
# Combine result["prompt_tokens"] and result["completion_tokens"] with the rates
# into an estimated cost, and print the token counts and the estimate.

In [24]:
INPUT_RATE = 0.075 / 1_000_000  # USD per input token (approximate)
OUTPUT_RATE = 0.30 / 1_000_000  # USD per output token (approximate)

input_tokens = result["prompt_tokens"]
output_tokens = result["completion_tokens"]

input_cost = input_tokens * INPUT_RATE
output_cost = output_tokens * OUTPUT_RATE
estimated_cost = input_cost + output_cost

print(f"Input tokens:       {input_tokens:,}")
print(f"Output tokens:      {output_tokens:,}")
print(f"Estimated input cost:  ${input_cost:.6f}")
print(f"Estimated output cost: ${output_cost:.6f}")
print(f"Estimated total cost:   ${estimated_cost:.6f}")

Input tokens:       1,090
Output tokens:      96
Estimated input cost:  $0.000082
Estimated output cost: $0.000029
Estimated total cost:   $0.000111


## Summary

When you are done, your notebook should:

- Build a SQL agent from scratch that inspects the schema and runs queries through the two tools.
- Assert on both the agent's answer and the order of its tool calls.
- Use a second model as a judge to grade an answer against a natural-language criterion.
- Track token usage and estimate the cost of a run.

The agent is the same hand-written loop from notebook 1, pointed at real tools over a real database.

## References & Further Reading

- [**Groq Tool Use**](https://console.groq.com/docs/tool-use): Function calling and the tool-call loop.
- [**DuckDB Documentation**](https://duckdb.org/docs/): Querying Parquet and running SQL in-process.
- [**NYC TLC Trip Record Data**](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page): The source of the taxi dataset.
- [**Groq Console**](https://console.groq.com/playground): Create a free API key and try the model used here.